## RFM Analysis
* > 1 - create (Recency , Frequency , Monetary)
* > 2 - create RFM Table
* > 3 - save RFM Table

* this use to :
Clustering  | 
Classification | 
ML models | 
Business insights | 


In [1]:
import pandas as pd


In [2]:
rfm_final_data = pd.read_csv("rfm_final_data.csv")
rfm_final_data['InvoiceDate'] = pd.to_datetime(rfm_final_data['InvoiceDate'])

rfm_final_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Total_price,InvoiceYear,InvoiceMonth,DayOfWeek,hour,weekend
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,Wednesday,8,0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,Wednesday,8,0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,Wednesday,8,0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,Wednesday,8,0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,Wednesday,8,0


In [3]:
rfm_final_data.dtypes

InvoiceNo                int64
StockCode               object
Description             object
Quantity                 int64
InvoiceDate     datetime64[ns]
UnitPrice              float64
CustomerID             float64
Country                 object
Total_price            float64
InvoiceYear              int64
InvoiceMonth             int64
DayOfWeek               object
hour                     int64
weekend                  int64
dtype: object

* > 1.1- create Monetary

In [4]:
Monetary = rfm_final_data.groupby(['CustomerID'])['Total_price'].sum().reset_index()
Monetary

,CustomerID,Total_price
0,12346.0,77183.60
1,12347.0,4310.00
2,12348.0,1797.24
3,12349.0,1757.55
4,12350.0,334.40
...,...,...
4333,18280.0,180.60
4334,18281.0,80.82
4335,18282.0,178.05
4336,18283.0,2045.53


* > 1.2- create Frequency

In [5]:
Frequency = (
    rfm_final_data
    .groupby('CustomerID')['InvoiceNo']
    .nunique()
    .reset_index()
    .rename(columns={'InvoiceNo':'Frequency'})
)
Frequency

'''I didn't use count because each product is stored with a different invoice number,
so I want to count the number of unique purchases, not the number of repeat purchases.'''

"I didn't use count because each product is stored with a different invoice number,\nso I want to count the number of unique purchases, not the number of repeat purchases."

* > 1.3- create Recency

In [6]:
last_date = rfm_final_data['InvoiceDate'].max()
print(last_date)
print(type(last_date))

2011-12-09 12:50:00
<class 'pandas._libs.tslibs.timestamps.Timestamp'>


In [9]:
Recency = (
    rfm_final_data
    .groupby('CustomerID')['InvoiceDate']
    .max()
    .reset_index()
)

Recency['Recency'] = (
    last_date - Recency['InvoiceDate']
).dt.days

In [ ]:
Recency

,CustomerID,InvoiceDate,Recency
0,12346.0,2011-01-18 10:01:00,325
1,12347.0,2011-12-07 15:52:00,1
2,12348.0,2011-09-25 13:13:00,74
3,12349.0,2011-11-21 09:51:00,18
4,12350.0,2011-02-02 16:01:00,309
...,...,...,...
4333,18280.0,2011-03-07 09:52:00,277
4334,18281.0,2011-06-12 10:53:00,180
4335,18282.0,2011-12-02 11:43:00,7
4336,18283.0,2011-12-06 12:02:00,3


* > 2 - create RFM Table

In [12]:
rfm = Recency.merge(Frequency, on='CustomerID')
rfm_table = rfm.merge(Monetary, on='CustomerID')
rfm_table.rename(columns={'Total_price':'Monetary'}, inplace=True)
rfm_table.drop(columns=['InvoiceDate'], inplace=True , axis=1)
rfm_table['CustomerID'] = rfm_table['CustomerID'].astype(int)
rfm_table

,CustomerID,Recency,Frequency,Monetary
0,12346,325,1,77183.60
1,12347,1,7,4310.00
2,12348,74,4,1797.24
3,12349,18,1,1757.55
4,12350,309,1,334.40
...,...,...,...,...
4333,18280,277,1,180.60
4334,18281,180,1,80.82
4335,18282,7,2,178.05
4336,18283,3,16,2045.53


* > 3 - save RFM Table

In [13]:
rfm_table.to_csv('rfm_table.csv', index=False)